# 🩺 HierDR-Net — Full MLOps Training Pipeline
**Two-Stage Hierarchical EfficientNet-B4 | Diabetic Retinopathy | APTOS 2019**

---
⚠️ **Before running:** Go to `Runtime → Change runtime type → T4 GPU → Save`

| Cell | What it does |
|---|---|
| 1 | Mount Drive + extract project |
| 2 | Install dependencies |
| 3 | Set working dir + verify GPU |
| 4 | Kaggle credentials |
| 5 | Point data to fast local storage |
| 6 | Run data ingestion |
| 7 | Run data validation |
| 8 | Train HierDRNet (full pipeline) |
| 9 | Train Cascade (full pipeline) |
| 10 | Evaluate both models |
| 11 | Save models to Drive |
| 12 | Download models to PC |


## 📁 Cell 1 — Mount Drive & Extract Project

In [1]:
# Install colabcode for VS Code connection
!pip install colabcode -q

In [2]:
from google.colab import drive
import zipfile, shutil, os

drive.mount('/content/drive')

# Remove any old version
shutil.rmtree('/content/drive/MyDrive/hierdr_fresh', ignore_errors=True)

# Extract fresh copy
zip_path = '/content/drive/MyDrive/hierdr-mlops.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/drive/MyDrive/')

print('✅ Project extracted')
print('📁 Contents:', os.listdir('/content/drive/MyDrive/hierdr_fresh'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Project extracted
📁 Contents: ['.github', 'config', 'data', 'logs', 'models', 'notebooks', 'reports', 'src', 'tests', 'params.yaml', 'requirements.txt', 'setup.py', '.env.example', 'Dockerfile', 'README.md']


## 📦 Cell 2 — Install Dependencies

In [3]:
import os

path = '/content/drive/MyDrive/hierdr_fresh'
print('Folder exists:', os.path.exists(path))
print('Contents:', os.listdir(path) if os.path.exists(path) else '❌ NOT FOUND')
print('requirements.txt:', os.path.exists(f'{path}/requirements.txt'))

Folder exists: True
Contents: ['.github', 'config', 'data', 'logs', 'models', 'notebooks', 'reports', 'src', 'tests', 'params.yaml', 'requirements.txt', 'setup.py', '.env.example', 'Dockerfile', 'README.md']
requirements.txt: True


In [4]:
import subprocess
subprocess.run(['pip', 'install', '-r',
    '/content/drive/MyDrive/hierdr_fresh/requirements.txt', '-q'], check=True)
subprocess.run(['pip', 'install', '-e',
    '/content/drive/MyDrive/hierdr_fresh', '-q'], check=True)
print('✅ All dependencies installed')

✅ All dependencies installed


## ⚙️ Cell 3 — Set Working Directory & Verify GPU

In [5]:
import os, sys, torch

PROJECT = '/content/drive/MyDrive/hierdr_fresh'
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print('📁 Working dir :', os.getcwd())
print('📦 src exists  :', os.path.exists('src'))
print('📄 train.py    :', os.path.exists('src/models/train.py'))
print('📄 cascade.py  :', os.path.exists('src/models/cascade_trainer.py'))
print('📄 pipeline    :', os.path.exists('src/pipelines/cascade_training_pipeline.py'))
print()
print('🖥️  CUDA        :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('🖥️  GPU         :', torch.cuda.get_device_name(0))
    print('🖥️  VRAM        :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
else:
    print('❌ No GPU — go to Runtime → Change runtime type → T4 GPU')

📁 Working dir : /content/drive/MyDrive/hierdr_fresh
📦 src exists  : True
📄 train.py    : True
📄 cascade.py  : True
📄 pipeline    : True

🖥️  CUDA        : True
🖥️  GPU         : Tesla T4
🖥️  VRAM        : 15.6 GB


## 🔑 Cell 4 — Kaggle Credentials

In [6]:
import os

# ⚠️ Replace with your actual credentials
# Get from: kaggle.com → Profile → Settings → API → Create New Token
os.environ['KAGGLE_USERNAME'] = 'Your_kaggle_user_name'
os.environ['KAGGLE_KEY']      = 'Your_key'

print('KAGGLE_USERNAME:', os.environ['KAGGLE_USERNAME'])
print('KAGGLE_KEY     : ****' + os.environ['KAGGLE_KEY'][-4:])

KAGGLE_USERNAME: blessingasare1
KAGGLE_KEY     : ****5b5c


## 🚄 Cell 5 — Point Data to Fast Local Storage

In [7]:
import yaml, os

with open('config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# /content/ is 10-25x faster than Drive for large image datasets
config['data_ingestion']['root_dir']       = '/content/artifacts/data_ingestion'
config['data_ingestion']['unzip_dir']      = '/content/artifacts/data_ingestion/data'
config['data_transformation']['data_path'] = '/content/artifacts/data_ingestion/data'

# Keep model checkpoints on Drive so they survive session disconnects
config['model_trainer']['trained_model_path']   = '/content/drive/MyDrive/hierdr-mlops-models/hierdr_net.pth'
config['cascade_model']['stage1_model_dir']     = '/content/drive/MyDrive/hierdr-mlops-models/cascade'
config['cascade_model']['stage1_model_path']    = '/content/drive/MyDrive/hierdr-mlops-models/cascade/stage1_net.pth'
config['cascade_model']['stage2_model_dir']     = '/content/drive/MyDrive/hierdr-mlops-models/cascade'
config['cascade_model']['stage2_model_path']    = '/content/drive/MyDrive/hierdr-mlops-models/cascade/stage2_net.pth'
config['model_evaluation']['metrics_path']      = '/content/drive/MyDrive/hierdr-mlops-models/eval_metrics.json'

with open('config/config.yaml', 'w') as f:
    yaml.dump(config, f)

os.makedirs('/content/artifacts/data_ingestion', exist_ok=True)
os.makedirs('/content/drive/MyDrive/hierdr-mlops-models/cascade', exist_ok=True)

print('✅ Config updated')
print('   Data   → /content/          (fast, 200-500 MB/s)')
print('   Models → Drive (safe, survives disconnects)')

✅ Config updated
   Data   → /content/          (fast, 200-500 MB/s)
   Models → Drive (safe, survives disconnects)


## ⬇️ Cell 6 — Data Ingestion (Download APTOS 2019)

In [8]:
import os
os.chdir('/content/drive/MyDrive/hierdr_fresh')

# Set Kaggle credentials
os.environ['KAGGLE_USERNAME'] = 'blessingasare1'
os.environ['KAGGLE_KEY']      = '43fe7d133a9103b765538a993aa65b5c'

from src.config.config import ConfigurationManager
from src.data.ingestion import DataIngestion

DataIngestion(ConfigurationManager().get_data_ingestion_config()).run()
print('✅ Data ingestion complete')

✅ Data ingestion complete


In [10]:
import os
print('CWD:', os.getcwd())
print('artifacts exists:', os.path.exists('artifacts'))
print('Drive project:', os.listdir('/content/drive/MyDrive/hierdr_fresh'))

CWD: /content/drive/MyDrive/hierdr_fresh
artifacts exists: True
Drive project: ['.github', 'config', 'data', 'logs', 'models', 'notebooks', 'reports', 'src', 'tests', 'params.yaml', 'requirements.txt', 'setup.py', '.env.example', 'Dockerfile', 'README.md', 'hierdr_mlops.egg-info', 'artifacts']


In [11]:
import os, sys

os.chdir('/content/drive/MyDrive/hierdr_fresh')
sys.path.insert(0, '/content/drive/MyDrive/hierdr_fresh')

os.environ['KAGGLE_USERNAME'] = 'your_kaggle_username'
os.environ['KAGGLE_KEY']      = 'your_kaggle_api_key'

# Create the folder manually first
os.makedirs('artifacts/data_ingestion/data', exist_ok=True)

from src.config.config import ConfigurationManager
from src.data.ingestion import DataIngestion

DataIngestion(ConfigurationManager().get_data_ingestion_config()).run()
print('✅ Done')

✅ Done


In [12]:
import os
os.makedirs('artifacts/data_ingestion/data', exist_ok=True)

os.environ['KAGGLE_USERNAME'] = 'blessingasare1'
os.environ['KAGGLE_KEY']      = '43fe7d133a9103b765538a993aa65b5c'

# Download directly via kaggle CLI
!kaggle competitions download -c aptos2019-blindness-detection -p artifacts/data_ingestion/data
!unzip -q artifacts/data_ingestion/data/aptos2019-blindness-detection.zip -d artifacts/data_ingestion/data/
!ls artifacts/data_ingestion/data/

100% 9.51G/9.51G [00:52<00:00, 196MB/s] 

aptos2019-blindness-detection.zip  test.csv	train.csv
sample_submission.csv		   test_images	train_images


In [13]:
import os
data_path = 'artifacts/data_ingestion/data'
print(os.listdir(data_path))
# Should show: ['train_images', 'train.csv', ...]

['aptos2019-blindness-detection.zip', 'sample_submission.csv', 'test.csv', 'test_images', 'train.csv', 'train_images']


In [14]:
import os, sys
os.chdir('/content/drive/MyDrive/hierdr_fresh')
sys.path.insert(0, '/content/drive/MyDrive/hierdr_fresh')

from src.config.config import ConfigurationManager
from src.data.ingestion import DataIngestion

DataIngestion(ConfigurationManager().get_data_ingestion_config()).run()
print('✅ Data ingestion complete')

✅ Data ingestion complete


## ✔️ Cell 7 — Data Validation

In [16]:
fix = '''import os, sys
from src.entity.config_entity import DataValidationConfig
from src.utils.logger import logger
from src.utils.exception import HierDRException


class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate(self) -> bool:
        try:
            os.makedirs(self.config.root_dir, exist_ok=True)
            base = os.path.dirname(self.config.status_file)
            os.makedirs(base, exist_ok=True)
            missing = []
            for f in self.config.required_files:
                full = os.path.join(
                    "artifacts/data_ingestion/data", f
                )
                if not os.path.exists(full):
                    missing.append(f)

            status = len(missing) == 0
            with open(self.config.status_file, "w") as sf:
                sf.write(f"Validation status: {status}\\n")
                if missing:
                    sf.write(f"Missing: {missing}\\n")

            if status:
                logger.info("Data validation passed.")
            else:
                logger.error(f"Data validation FAILED. Missing: {missing}")
            return status
        except Exception as e:
            raise HierDRException(e, sys)


if __name__ == "__main__":
    from src.config.config import ConfigurationManager
    ok = DataValidation(ConfigurationManager().get_data_validation_config()).validate()
    if not ok:
        raise SystemExit("Data validation failed — check your dataset.")
'''

with open('/content/drive/MyDrive/hierdr_fresh/src/data/validation.py', 'w') as f:
    f.write(fix)

print('✅ validation.py fixed')

✅ validation.py fixed


In [17]:
from src.config.config import ConfigurationManager
from src.data.validation import DataValidation

ok = DataValidation(ConfigurationManager().get_data_validation_config()).validate()
if not ok:
    raise Exception('❌ Validation failed — check your dataset folder')
print('✅ Validation passed — ready to train')

✅ Validation passed — ready to train


## 🚀 Cell 8 — Train HierDRNet (Shared Backbone + Hierarchical Loss)

In [18]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()

import os, sys
os.chdir('/content/drive/MyDrive/hierdr_fresh')
sys.path.insert(0, '/content/drive/MyDrive/hierdr_fresh')

from src.config.config import ConfigurationManager
from src.data.transformation import DataTransformation
from src.models.train import ModelTrainer

manager = ConfigurationManager()
train_loader, val_loader, _ = DataTransformation(
    manager.get_data_transformation_config()
).get_data_loaders()

print('HierDRNet Training')
print('Backbone  : EfficientNet-B4')
print('Loss      : Hierarchical (Stage1 binary + Stage2 severity masked)')
print('Precision : Mixed (AMP) — saves ~40% VRAM')
print('-' * 50)

ModelTrainer(manager.get_model_trainer_config()).train(train_loader, val_loader)
print('\n✅ HierDRNet training complete')

HierDRNet Training
Backbone  : EfficientNet-B4
Loss      : Hierarchical (Stage1 binary + Stage2 severity masked)
Precision : Mixed (AMP) — saves ~40% VRAM
--------------------------------------------------


/content/drive/MyDrive/hierdr_fresh/src/models/train.py:55: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()
/content/drive/MyDrive/hierdr_fresh/src/models/train.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/hierdr_fresh/src/models/train.py:88: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



✅ HierDRNet training complete


## 🚀 Cell 9 — Train Full Cascade (Stage1Net → Stage2Net)

In [19]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()

import os, sys
os.chdir('/content/drive/MyDrive/hierdr_fresh')
sys.path.insert(0, '/content/drive/MyDrive/hierdr_fresh')

from src.config.config import ConfigurationManager
from src.data.transformation import DataTransformation
from src.models.cascade_trainer import CascadeTrainer

manager = ConfigurationManager()
train_loader, val_loader, _ = DataTransformation(
    manager.get_data_transformation_config()
).get_data_loaders()

print('Cascade Training')
print('Stage 1   : Stage1Net — binary DR / No-DR (all images)')
print('Stage 2   : Stage2Net — severity grading (DR-positive only)')
print('Signal    : Stage1 DR probability fed as extra input to Stage2')
print('-' * 50)

CascadeTrainer(manager.get_cascade_config()).train(train_loader, val_loader)
print('\n✅ Cascade training complete')

Cascade Training
Stage 1   : Stage1Net — binary DR / No-DR (all images)
Stage 2   : Stage2Net — severity grading (DR-positive only)
Signal    : Stage1 DR probability fed as extra input to Stage2
--------------------------------------------------


/content/drive/MyDrive/hierdr_fresh/src/models/cascade_trainer.py:27: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()
/content/drive/MyDrive/hierdr_fresh/src/models/cascade_trainer.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/hierdr_fresh/src/models/cascade_trainer.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/hierdr_fresh/src/models/cascade_trainer.py:83: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()
/content/drive/MyDrive/hierdr_fresh/src/models/cascade_trainer.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is depr


✅ Cascade training complete


## 📊 Cell 10 — Evaluate Both Models

In [20]:
import os, sys
os.chdir('/content/drive/MyDrive/hierdr_fresh')
sys.path.insert(0, '/content/drive/MyDrive/hierdr_fresh')

from src.config.config import ConfigurationManager
from src.data.transformation import DataTransformation
from src.models.evaluate import ModelEvaluator

manager = ConfigurationManager()
_, _, test_loader = DataTransformation(
    manager.get_data_transformation_config()
).get_data_loaders()

evaluator = ModelEvaluator(
    manager.get_model_evaluation_config(),
    manager.get_model_trainer_config(),
    manager.get_cascade_config(),
)
metrics = evaluator.evaluate(test_loader)

print('\n📊 FINAL RESULTS')
print('=' * 45)
for name, m in metrics.items():
    print(f"  {name:<12}  Accuracy: {m['accuracy']}   Kappa: {m['quadratic_kappa']}")
print('=' * 45)
print('Metrics saved to Drive ✅')

/content/drive/MyDrive/hierdr_fresh/src/models/evaluate.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/hierdr_fresh/src/models/evaluate.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/hierdr_fresh/src/models/evaluate.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



📊 FINAL RESULTS
  hierdrnet     Accuracy: 0.812   Kappa: 0.8559
  cascade       Accuracy: 0.7956   Kappa: 0.8181
Metrics saved to Drive ✅


## 💾 Cell 11 — Verify Models Saved on Drive

In [21]:
import os

DRIVE_MODELS = '/content/drive/MyDrive/hierdr-mlops-models'
print('📁 Saved models:')
for root, dirs, files in os.walk(DRIVE_MODELS):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1e6
        print(f'   {f:<35} {size:.1f} MB')

📁 Saved models:
   hierdr_net.pth                      74.6 MB
   eval_metrics.json                   0.0 MB
   stage1_net.pth                      72.8 MB
   stage2_net.pth                      72.8 MB


## ⬇️ Cell 12 — Download Models to Your PC

In [22]:
from google.colab import files
import os

DRIVE_MODELS = '/content/drive/MyDrive/hierdr-mlops-models'

to_download = [
    f'{DRIVE_MODELS}/hierdr_net.pth',
    f'{DRIVE_MODELS}/cascade/stage1_net.pth',
    f'{DRIVE_MODELS}/cascade/stage2_net.pth',
    f'{DRIVE_MODELS}/eval_metrics.json',
]

for path in to_download:
    if os.path.exists(path):
        files.download(path)
        print(f'⬇️  {os.path.basename(path)}')
    else:
        print(f'⚠️  Not found (not trained yet?): {path}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  hierdr_net.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

⬇️  stage2_net.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  eval_metrics.json


In [6]:
pip install mlflow

In [11]:
import json, mlflow

# Load metrics from saved file
with open('/content/drive/MyDrive/hierdr-mlops-models/eval_metrics.json') as f:
    metrics = json.load(f)

print("📊 Loaded metrics:", metrics)

# Log to MLflow (keys are lowercase: "hierdrnet", "cascade")
with mlflow.start_run(run_name="HierDRNet_eval"):
    mlflow.log_metric("hierdrnet_accuracy",  metrics["hierdrnet"]["accuracy"])
    mlflow.log_metric("hierdrnet_kappa",     metrics["hierdrnet"]["quadratic_kappa"])
    mlflow.log_metric("cascade_accuracy",    metrics["cascade"]["accuracy"])
    mlflow.log_metric("cascade_kappa",       metrics["cascade"]["quadratic_kappa"])
    mlflow.log_artifact("/content/drive/MyDrive/hierdr-mlops-models/eval_metrics.json")

print("✅ Metrics logged to MLflow")

📊 Loaded metrics: {'hierdrnet': {'accuracy': 0.812, 'quadratic_kappa': 0.8559}, 'cascade': {'accuracy': 0.7956, 'quadratic_kappa': 0.8181}}
✅ Metrics logged to MLflow


In [15]:
# # Install ngrok
# !pip install pyngrok -q

# from pyngrok import ngrok
# import subprocess, threading

# # Start MLflow server in background
# subprocess.Popen(['mlflow', 'ui', '--port', '5000',
#                   '--host', '0.0.0.0'])

# # Create public tunnel
# public_url = ngrok.connect(5000)
# print(f"🔗 MLflow Dashboard: {public_url}")

In [14]:
import json

with open('/content/drive/MyDrive/hierdr-mlops-models/eval_metrics.json') as f:
    metrics = json.load(f)

print("\n📊 EVALUATION RESULTS")
print("=" * 45)
for model, m in metrics.items():
    print(f"\n  {model}")
    print(f"    Accuracy        : {m['accuracy']:.4f}")
    print(f"    Quadratic Kappa : {m['quadratic_kappa']:.4f}")
print("=" * 45)


📊 EVALUATION RESULTS

  hierdrnet
    Accuracy        : 0.8120
    Quadratic Kappa : 0.8559

  cascade
    Accuracy        : 0.7956
    Quadratic Kappa : 0.8181
